# LightGBM LambdaRank Reranker (GPU + Optuna + F₀.₅ Calibration + ROC Curves)
Optimized for Precision-heavy Metric (F0.5) with Singleton Handling.


In [ ]:
import os
import gc
import time
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve

try:
    import lightgbm as lgb
except ImportError:
    os.system("pip install -q lightgbm")
    import lightgbm as lgb

try:
    import optuna
except ImportError:
    os.system("pip install -q optuna")
    import optuna

try:
    from tqdm.auto import tqdm
except ImportError:
    os.system("pip install -q tqdm")
    from tqdm.auto import tqdm

import joblib

DATA_PATH = Path('/kaggle/working/training_features_full.csv')
OUTPUT_DIR = Path('/kaggle/working/lgbm_reranker')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
print("[INFO] Loading reranker training dataset...")
df = pd.read_csv(DATA_PATH, dtype={"query_id": str, "candidate_id": str})

FEATURE_COLUMNS = [
    'exact_name_norm', 'exact_name_ascii', 'fuzzy_ratio', 'fuzzy_partial_ratio', 
    'fuzzy_token_sort_ratio', 'fuzzy_token_set_ratio', 'fuzzy_WRatio', 'jaro_winkler', 
    'token_overlap', 'token_jaccard', 'token_containment', 'char_2gram_similarity', 
    'char_3gram_similarity', 'char_4gram_similarity', 'char_5gram_similarity', 
    'name_length_difference', 'name_length_ratio', 'name_token_count_diff', 
    'first_token_match', 'last_token_match', 'address_exact_match', 'address_fuzzy_ratio', 
    'address_partial_ratio', 'address_length_difference', 'address_number_match', 
    'legal_form_missing', 'both_legal_form_present', 'one_legal_form_present', 
    'same_legal_form', 'legal_form_match', 'country_equal', 'ann_rank', 'ann_similarity'
]

missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing:
    print(f"[WARN] Missing columns: {missing}")
FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c in df.columns]

df["label"] = df["label"].astype(float).astype(int)
for col in FEATURE_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

print(f"Shape: {df.shape} | Features: {len(FEATURE_COLUMNS)}")


In [ ]:
def query_partitioned_split(dataset_df, val_fraction=0.2, random_state=42):
    unique_queries = np.array(sorted(dataset_df["query_id"].unique()))
    rng = np.random.default_rng(random_state)
    rng.shuffle(unique_queries)
    val_size = max(1, int(len(unique_queries) * val_fraction))
    val_query_set = set(unique_queries[:val_size])
    train_df = dataset_df[~dataset_df["query_id"].isin(val_query_set)].copy()
    val_df = dataset_df[dataset_df["query_id"].isin(val_query_set)].copy()
    train_df = train_df.sort_values("query_id").reset_index(drop=True)
    val_df = val_df.sort_values("query_id").reset_index(drop=True)
    return train_df, val_df

def compute_group_sizes(sorted_df):
    return sorted_df.groupby("query_id", sort=False).size().to_numpy(dtype=np.int32)

train_df, val_df = query_partitioned_split(df, val_fraction=0.2)
train_groups = compute_group_sizes(train_df)
val_groups = compute_group_sizes(val_df)

X_train, y_train = train_df[FEATURE_COLUMNS], train_df["label"].to_numpy(dtype=np.int32)
X_val, y_val = val_df[FEATURE_COLUMNS], val_df["label"].to_numpy(dtype=np.int32)


In [ ]:
# Pre-compute Ground Truth Dicts for Evaluation
val_y_true_by_query = val_df[val_df['label'] == 1].groupby('query_id')['candidate_id'].apply(set).to_dict()
val_all_queries = set(val_df['query_id'].unique())

def macro_f05(y_true_by_query, pred_by_query, all_queries):
    scores = []
    for qid in all_queries:
        true_set = y_true_by_query.get(qid, set())
        pred_set = set(pred_by_query.get(qid, []))
        
        if len(true_set) == 0:
            score = 1.0 if len(pred_set) == 0 else 0.0
        else:
            tp = len(true_set & pred_set)
            precision = tp / len(pred_set) if pred_set else 0.0
            recall = tp / len(true_set)
            if precision == 0 and recall == 0:
                score = 0.0
            else:
                score = (1.25 * precision * recall) / (0.25 * precision + recall)
        scores.append(score)
    return np.mean(scores)

def search_best_threshold(val_preds_df):
    """Fast threshold search inside Optuna"""
    val_preds_df = val_preds_df.sort_values(["query_id", "score"], ascending=[True, False])
    val_preds_df['rank'] = val_preds_df.groupby('query_id').cumcount() + 1
    val_preds_df['top_score'] = val_preds_df.groupby('query_id')['score'].transform('max')
    
    best_score = -1
    
    # Margin Thresholds (Relative to Top 1)
    for margin in [0.0, 0.5, 1.0, 2.0, 3.0]:
        valid_preds = val_preds_df[(val_preds_df['score'] >= (val_preds_df['top_score'] - margin))]
        pred_dict = valid_preds.groupby("query_id")["candidate_id"].apply(list).to_dict()
        score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
        if score > best_score: best_score = score
            
    # Global Absolute Thresholds
    unique_scores = np.sort(val_preds_df['score'].unique())[::-1]
    test_thresh = np.percentile(unique_scores, np.linspace(0, 100, 20)) # 20 percentiles
    
    for t in test_thresh:
        for k in [3, 5, 10, 20]:
            valid_preds = val_preds_df[(val_preds_df['score'] >= t) & (val_preds_df['rank'] <= k)]
            pred_dict = valid_preds.groupby("query_id")["candidate_id"].apply(list).to_dict()
            score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
            if score > best_score: best_score = score
                
    return best_score


In [ ]:
def objective(trial):
    params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "device_type": "gpu",  # <--- GPU ENABLED FOR KAGGLE
        "lambdarank_truncation_level": trial.suggest_categorical("lambdarank_truncation_level", [10, 20, 50]),
        "ndcg_at": [1, 3, 5, 10, 20],
        "learning_rate": trial.suggest_categorical("learning_rate", [0.02, 0.03, 0.05]),
        "n_estimators": 500,
        "num_leaves": trial.suggest_categorical("num_leaves", [15, 31, 63]),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 6, 10]),
        "min_child_samples": trial.suggest_categorical("min_child_samples", [2, 5, 10]),
        "feature_fraction": trial.suggest_categorical("feature_fraction", [0.7, 0.85, 1.0]),
        "bagging_fraction": trial.suggest_categorical("bagging_fraction", [0.7, 0.85, 1.0]),
        "lambda_l1": trial.suggest_categorical("lambda_l1", [0, 0.1, 1.0]),
        "lambda_l2": trial.suggest_categorical("lambda_l2", [0.1, 1.0, 5.0]),
        "n_jobs": -1,
        "random_state": 42,
        "verbose": -1
    }
    
    ranker = lgb.LGBMRanker(**params)
    
    callbacks = [lgb.early_stopping(stopping_rounds=30, verbose=False)]
    ranker.fit(
        X=X_train, y=y_train, group=train_groups,
        eval_set=[(X_val, y_val)], eval_group=[val_groups],
        callbacks=callbacks
    )
    
    # Predict Validation
    val_preds = val_df[['query_id', 'candidate_id', 'label']].copy()
    val_preds['score'] = ranker.predict(X_val)
    
    # Search Thresholds
    best_f05 = search_best_threshold(val_preds)
    return best_f05

print("[INFO] Starting Optuna Hyperparameter Optimization on GPU...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=True) # Full 20 Trials

print(f"\n🏆 Best Macro F0.5: {study.best_value:.5f}")
print(f"🏆 Best Params: {study.best_params}")


In [ ]:
# === RETRAIN BEST MODEL ===
best_params = study.best_params
best_params.update({
    "objective": "lambdarank",
    "metric": "ndcg",
    "device_type": "gpu",
    "ndcg_at": [1, 3, 5, 10, 20],
    "n_estimators": 1000,
    "n_jobs": -1,
    "random_state": 42
})

print("[INFO] Retraining final model with best params on GPU...")
final_ranker = lgb.LGBMRanker(**best_params)
final_ranker.fit(
    X=X_train, y=y_train, group=train_groups,
    eval_set=[(X_val, y_val)], eval_group=[val_groups],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

# ========================================================
# EXPLICITLY SAVE WEIGHTS RIGHT AFTER TRAINING IN SAME CELL
# ========================================================
weights_path = OUTPUT_DIR / "lgbm_model_weights.txt"
final_ranker.booster_.save_model(str(weights_path))
print(f"\n[SUCCESS] Model Weights successfully saved to: {weights_path}\n")

# Detailed Threshold Calibration on Best Model
val_preds = val_df[['query_id', 'candidate_id', 'label']].copy()
val_preds['score'] = final_ranker.predict(X_val)

def exhaustive_threshold_search(val_preds_df):
    print("[INFO] Running Exhaustive Threshold Sweep...")
    val_preds_df = val_preds_df.sort_values(["query_id", "score"], ascending=[True, False])
    val_preds_df['rank'] = val_preds_df.groupby('query_id').cumcount() + 1
    val_preds_df['top_score'] = val_preds_df.groupby('query_id')['score'].transform('max')
    
    best_f05 = -1
    best_config = {}
    
    # Margin Sweep with TQDM
    for margin in tqdm(np.arange(0.0, 5.0, 0.1), desc="Margin Sweep"):
        valid_preds = val_preds_df[(val_preds_df['score'] >= (val_preds_df['top_score'] - margin))]
        pred_dict = valid_preds.groupby("query_id")["candidate_id"].apply(list).to_dict()
        score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
        if score > best_f05: 
            best_f05 = score
            best_config = {'type': 'margin', 'margin': margin}
            
    # Global + Rank Sweep with TQDM
    unique_scores = np.sort(val_preds_df['score'].unique())[::-1]
    test_thresh = np.percentile(unique_scores, np.linspace(0, 100, 100))
    for t in tqdm(test_thresh, desc="Global Sweep"):
        for k in [1, 2, 3, 5, 10, 20]:
            valid_preds = val_preds_df[(val_preds_df['score'] >= t) & (val_preds_df['rank'] <= k)]
            pred_dict = valid_preds.groupby("query_id")["candidate_id"].apply(list).to_dict()
            score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
            if score > best_f05:
                best_f05 = score
                best_config = {'type': 'global_k', 'threshold': t, 'max_k': k}
                
    return best_f05, best_config

final_f05, final_thresh_config = exhaustive_threshold_search(val_preds)

print("\n" + "="*60)
print("FINAL OPTIMIZED PIPELINE")
print("="*60)
print(f"Macro F0.5    : {final_f05:.6f}")
print(f"Threshold Strategy : {final_thresh_config}")

# Save Everything including Joblib Pipeline
model_path = OUTPUT_DIR / "lgbm_optimized_reranker.joblib"
joblib.dump({
    "model": final_ranker,
    "features": FEATURE_COLUMNS,
    "best_params": best_params,
    "threshold_config": final_thresh_config,
    "val_macro_f05": final_f05
}, model_path)
print(f"[SUCCESS] Full Pipeline Saved to {model_path}")


In [ ]:
# ==============================================================================
# PLOT ROC AUC AND PRECISION-RECALL CURVES
# ==============================================================================
print("[INFO] Generating ROC and Precision-Recall Curves...")

# Ground truth labels (1 or 0) and predicted ranking scores
y_true = val_preds['label'].values
y_scores = val_preds['score'].values

# Calculate curves
fpr, tpr, roc_thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

precision, recall, pr_thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. ROC Curve
ax1.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([-0.05, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
ax1.legend(loc="lower right", fontsize=12)
ax1.grid(True, alpha=0.3)

# 2. Precision-Recall Curve (Crucial for F0.5)
ax2.plot(recall, precision, color='blue', lw=2.5, label=f'PR curve (AUC = {pr_auc:.4f})')
ax2.set_xlim([-0.05, 1.05])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve (Crucial for F0.5)', fontsize=14, fontweight='bold')
ax2.legend(loc="lower left", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_curves.png")
print(f"\n[SUMMARY] Area Under ROC Curve (AUC) : {roc_auc:.4f}")
print(f"[SUMMARY] Area Under PR Curve (AUC)  : {pr_auc:.4f}")
print(f"[SUCCESS] Plot saved to {OUTPUT_DIR / 'roc_pr_curves.png'}")
